# Age, ageing and job reallocation (v1)

Do older or faster-ageing regions have less job reallocation? Uses the LBD job flows (`clean/itl2_2021/job_flows_itl2.csv`) and population by age (`clean/itl2_2021/age_itl2.csv`, `age_single_year_itl2.csv`; built in `build/age.ipynb` from ONS/NRS/NISRA mid-year estimates), on ITL2 **2021** (41 regions), with reallocation **pooled over 2011–2024** as in the rurality analysis.

**Why it might matter (two channels in the literature):**
1. **Labour-supply growth → firm entry.** Slower growth of the working-age population reduces start-up rates, and with them churn (Karahan, Pugsley & Şahin 2024, *AER*; Hopenhayn, Neira & Singhania 2022, *Econometrica*).
2. **Older workforce → less mobility.** Older workers change jobs less and firms adjust less through hiring and separations, which lowers reallocation (Engbom 2019).

**Measures (all from population counts summed over 2011–24, then derived at ITL2):**

| measure | what |
|---|---|
| Median age | median of the pooled single-year-of-age distribution |
| Share 65+ | 65+ as % of population |
| Prime-age share | 25–49 as % of population |
| Working-age growth | annual growth of the 16–64 population, 2011→2024 (log, % per year) |
| Ageing pace | change in median age, 2011→2024 (years) |

**The main pitfall is confounding.** Age is tightly linked to density (rural regions are older; London is very young), and the rurality analysis showed density predicts churn. Working-age growth is linked to **job growth** (jobs attract workers and vice versa), and reallocation includes net job growth by construction. So each correlation is shown:
- **raw**, **excluding London**, and with **Spearman** ranks;
- **controlling for density** (log population-weighted density);
- **controlling for density and net job growth**;
- for **excess reallocation** (churn beyond what net growth requires) and separately for **entry/exit** vs **incumbent** flows (the two channels above).

Descriptive, not causal: about 40 regions, and age, density and growth move together. Charts are saved to `analysis/demographics/charts/`; shared chart setup is in `analysis/chart_setup.py`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats

BASE  = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
DIR21 = BASE / "clean" / "itl2_2021"
FIG   = BASE / "analysis" / "demographics" / "charts"
FIG.mkdir(parents=True, exist_ok=True)
sys.path.append(str(BASE / "analysis"))
from chart_setup import styles, alt, ECO, TXT, GREY, place_labels

SOURCE = "Sources: ONS Longitudinal Business Database (Secure Access, SN 9501); ONS, NRS and NISRA mid-year population estimates; authors' calculations."
MAIN = (2011, 2024)
FLOWS = ["jc_entry", "jd_exit", "jc_incumbent_ext", "jd_incumbent_ext", "jc_incumbent_int", "jd_incumbent_int"]
LONDON = lambda d: d.itl2_code.str.startswith("TLI")

# --- reallocation, pooled 2011-24 from counts ---
reg = pd.read_csv(DIR21 / "job_flows_itl2.csv")
jf = reg[reg.analysis_window & ~((reg.itl2_code == "TLN0") & (reg.year < 2011))].copy()
g = jf[jf.year.between(*MAIN)].groupby("itl2_code")[FLOWS + ["total_jc", "total_jd", "emp_avg"]].sum()
y = pd.DataFrame({"realloc": 100 * (g.total_jc + g.total_jd) / g.emp_avg,
                  "net": 100 * (g.total_jc - g.total_jd) / g.emp_avg,
                  "entry_exit": 100 * (g.jc_entry + g.jd_exit) / g.emp_avg,
                  "incumbent": 100 * g[FLOWS[2:]].sum(axis=1) / g.emp_avg})
y["excess"] = y.realloc - y.net.abs()

# --- age measures from counts summed over 2011-24 ---
a = pd.read_csv(DIR21 / "age_itl2.csv").query("year >= @MAIN[0] and year <= @MAIN[1]")
s = a.groupby("itl2_code")[["population", "pop_25_49", "pop_65_plus", "pop_16_64"]].sum()
X = pd.DataFrame({"share_65_plus": 100 * s.pop_65_plus / s.population, "share_25_49": 100 * s.pop_25_49 / s.population})

sy = pd.read_csv(DIR21 / "age_single_year_itl2.csv").query("year >= @MAIN[0] and year <= @MAIN[1]")
def pooled_median(d):
    """Median of the pooled single-year distribution; age a covers [a, a+1), linear within the year."""
    c = d.groupby("age").persons.sum().sort_index(); cum = c.cumsum(); half = c.sum() / 2
    a_ = (cum >= half).idxmax(); below = cum.shift(fill_value=0)[a_]
    return a_ + (half - below) / c[a_]
X["median_age"] = sy.groupby("itl2_code").apply(pooled_median, include_groups=False)

wide = lambda col: a.pivot(index="itl2_code", columns="year", values=col)
X["wa_growth"] = 100 * np.log(wide("pop_16_64")[MAIN[1]] / wide("pop_16_64")[MAIN[0]]) / (MAIN[1] - MAIN[0])
X["ageing"] = wide("median_age")[MAIN[1]] - wide("median_age")[MAIN[0]]

# --- control: density ---
pwd = pd.read_csv(DIR21 / "pop_weighted_density_itl2.csv").query("year == 2021").set_index("itl2_code").pop_weighted_density
names = pd.read_csv(DIR21 / "geography_spine.csv").drop_duplicates("itl2_code").set_index("itl2_code").itl2_name
df = y.join(X).join(np.log(pwd).rename("log_pwd")).join(names).reset_index()
df["group"] = np.where(LONDON(df), "London", "Rest of UK")
assert len(df) == 41 and df.notna().all().all()
print(df[["median_age", "share_65_plus", "share_25_49", "wa_growth", "ageing"]].describe().round(2).loc[["mean", "min", "max"]].to_string())

      median_age  share_65_plus  share_25_49  wa_growth  ageing
mean       41.36          18.74        32.85       0.38    1.52
min        32.56           8.02        28.13      -0.40   -0.11
max        46.82          24.43        46.57       1.13    4.38


## 1. Correlations, and what survives the controls

The partial correlation removes the linear effect of the controls from both variables (multiple regression residuals). Samples exclude London unless stated. Correlation between each age measure and density is shown, since it signals how much confounding to expect.

In [2]:
def partial(dd, x, yv, controls):
    """Correlation of dd[x] and dd[yv] after regressing both on `controls` (plus a constant)."""
    Z = np.column_stack([np.ones(len(dd))] + [dd[c].values for c in controls])
    res = lambda v: v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]
    return np.corrcoef(res(dd[x].values), res(dd[yv].values))[0, 1]

MEAS = {"median_age": "Median age", "share_65_plus": "Share 65+", "share_25_49": "Prime-age share (25–49)",
        "wa_growth": "Working-age growth", "ageing": "Ageing pace (Δ median age)"}
o = df[~LONDON(df)]
rows = []
for m, lab in MEAS.items():
    rows.append({"measure": lab, "r all": np.corrcoef(df.realloc, df[m])[0, 1], "r excl. London": np.corrcoef(o.realloc, o[m])[0, 1],
                 "rho excl. London": stats.spearmanr(o.realloc, o[m])[0], "r with density": np.corrcoef(o[m], o.log_pwd)[0, 1],
                 "partial | density": partial(o, m, "realloc", ["log_pwd"]),
                 "partial | density + net growth": partial(o, m, "realloc", ["log_pwd", "net"]),
                 "excess realloc, r": np.corrcoef(o.excess, o[m])[0, 1],
                 "entry/exit, r": np.corrcoef(o.entry_exit, o[m])[0, 1], "incumbent, r": np.corrcoef(o.incumbent, o[m])[0, 1]})
corr = pd.DataFrame(rows)
print("Reallocation (pooled 2011–24) vs age measures; 'excl. London' samples have 36 regions")
print(corr.round(2).to_string(index=False))
print(f"\nNet job growth vs working-age growth (excl. London): r = {np.corrcoef(o.net, o.wa_growth)[0, 1]:.2f}")

# ageing within regions: did faster-ageing regions see bigger falls in churn?
def realloc_rate(yrs):
    gg = jf[jf.year.between(*yrs)].groupby("itl2_code")[["total_jc", "total_jd", "emp_avg"]].sum()
    return 100 * (gg.total_jc + gg.total_jd) / gg.emp_avg
df["d_realloc"] = df.itl2_code.map(realloc_rate((2021, 2024)) - realloc_rate((2011, 2014)))
o = df[~LONDON(df)]
print(f"Change in reallocation (2011–14 → 2021–24) vs ageing pace: r = {np.corrcoef(df.d_realloc, df.ageing)[0, 1]:+.2f} all, "
      f"{np.corrcoef(o.d_realloc, o.ageing)[0, 1]:+.2f} excl. London")

Reallocation (pooled 2011–24) vs age measures; 'excl. London' samples have 36 regions
                   measure  r all  r excl. London  rho excl. London  r with density  partial | density  partial | density + net growth  excess realloc, r  entry/exit, r  incumbent, r
                Median age  -0.79           -0.64             -0.59           -0.79              -0.34                           -0.13              -0.64          -0.33         -0.71
                 Share 65+  -0.80           -0.60             -0.61           -0.77              -0.27                           -0.17              -0.61          -0.29         -0.68
   Prime-age share (25–49)   0.85            0.63              0.61            0.76               0.34                            0.21               0.64           0.34          0.69
        Working-age growth   0.65            0.72              0.66            0.48               0.61                            0.05               0.66           0.72          0.60

## 2. Charts

**(a) Median age:** older regions churn less, but much of this reflects density. **(b) Working-age population growth:** the strongest age-related link, especially with firm entry and exit. **(c) Robustness summary:** how each correlation changes as controls are added.

In [3]:
W, H = 520, 380
uk = 100 * jf[jf.year.between(*MAIN)][["total_jc", "total_jd"]].sum().sum() / jf[jf.year.between(*MAIN)].emp_avg.sum()

def scatter(xcol, xtitle, title, sub, fname, xfmt=".1f", xpad=0.5, step=1.0):
    d = df.copy()
    XS = alt.Scale(domain=[float(np.floor((d[xcol].min() - xpad) / step) * step), float(np.ceil((d[xcol].max() + xpad) / step) * step)], zero=False, nice=False)
    YS = alt.Scale(domain=[float(np.floor(d.realloc.min() - 1)), float(np.ceil(d.realloc.max() + 1))], zero=False, nice=False)
    b = np.polyfit(d[xcol], d.realloc, 1); fx = np.linspace(d[xcol].min(), d[xcol].max(), 2)
    labs = place_labels(d, xcol, "realloc", XS, YS, W, H, max_labels=6, max_len=22)
    ch = alt.layer(
        alt.Chart(pd.DataFrame({"v": [uk]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS)),
        alt.Chart(pd.DataFrame({"x": [XS.domain[1]], "y": [uk]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=10, color=TXT)
           .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS), text=alt.value(f"UK {uk:.1f}%")),
        alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=ECO["turquoise"], strokeWidth=2, strokeDash=[6, 4])
           .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)),
        alt.Chart(d).mark_circle(size=70, opacity=0.85, stroke="white", strokeWidth=0.6).encode(
            x=alt.X(f"{xcol}:Q", scale=XS, title=xtitle), y=alt.Y("realloc:Q", scale=YS, title="Reallocation rate, 2011–24 (% of employment)"),
            color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                            legend=alt.Legend(title=None, orient="bottom")),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip(f"{xcol}:Q", format=xfmt, title=xtitle),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
            .encode(x=alt.X(f"{xcol}:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
    ).properties(title=alt.Title(title, subtitle=sub + [SOURCE]))
    styles.save(ch, str(FIG), fname, width=W, height=H)
    return ch.properties(width=W, height=H)

c = corr.set_index("measure")
sc_age = scatter("median_age", "Median age, 2011–24 (years)", "Older regions have less job reallocation, but much of this is density",
    [f"ITL2 regions, pooled 2011–24. Correlation r = {c.loc['Median age', 'r all']:.2f} all regions; {c.loc['Median age', 'r excl. London']:.2f} excl. London; "
     f"{c.loc['Median age', 'partial | density']:.2f} excl. London after controlling for density.",
     "Median age of the population pooled over 2011–24 (single-year mid-year estimates). Dashed line = fit."], "reallocation_median_age")
sc_age

alt.LayerChart(...)

In [4]:
sc_wa = scatter("wa_growth", "Working-age (16–64) population growth, 2011–24 (% per year)",
    "Regions with faster-growing working-age populations have more churn",
    [f"ITL2 regions, pooled 2011–24. r = {c.loc['Working-age growth', 'r all']:.2f} all regions; {c.loc['Working-age growth', 'r excl. London']:.2f} excl. London; "
     f"{c.loc['Working-age growth', 'partial | density']:.2f} after controlling for density, {c.loc['Working-age growth', 'partial | density + net growth']:.2f} also for net job growth.",
     "Annual log growth of the 16–64 population, 2011 to 2024. Dashed line = fit."], "reallocation_working_age_growth", xfmt=".2f", xpad=0.05, step=0.2)
sc_wa

alt.LayerChart(...)

In [5]:
# (c) robustness summary: correlations excl. London, raw -> controlling density -> density + net job growth
STEPS = {"r excl. London": "No controls", "partial | density": "Controlling for density",
         "partial | density + net growth": "Controlling for density and net job growth"}
rob = corr.melt(id_vars="measure", value_vars=list(STEPS), var_name="step", value_name="r")
rob["step"] = rob.step.map(STEPS)
MORDER = list(MEAS.values()); SORDER = list(STEPS.values())
SCOLS = [ECO["blue-dark"], ECO["blue-light"], ECO["orange"]]
dots = alt.layer(
    alt.Chart(pd.DataFrame({"v": [0.0]})).mark_rule(color=TXT, opacity=0.6).encode(x="v:Q"),
    alt.Chart(rob).mark_line(color=GREY, strokeWidth=1.2).encode(
        x="r:Q", y=alt.Y("measure:N", sort=MORDER), detail="measure:N"),
    alt.Chart(rob).mark_circle(size=110, opacity=1).encode(
        x=alt.X("r:Q", scale=alt.Scale(domain=[-1, 1]), title="Correlation with reallocation rate, 2011–24 (excl. London)"),
        y=alt.Y("measure:N", sort=MORDER, title=None),
        color=alt.Color("step:N", scale=alt.Scale(domain=SORDER, range=SCOLS), legend=alt.Legend(title=None, orient="bottom", labelLimit=400)),
        tooltip=["measure:N", "step:N", alt.Tooltip("r:Q", format=".2f")]),
).properties(title=alt.Title("Much of the age–churn link runs through density, and working-age growth's through job growth",
    subtitle=["Correlation between each age measure and the reallocation rate across 36 ITL2 regions (London excluded), pooled 2011–24.",
              "Partial correlations remove the linear effect of log population-weighted density, then also net job growth.", SOURCE]))
styles.save(dots, str(FIG), "reallocation_age_robustness", width=520, height=230)
dots.properties(width=520, height=230)

alt.LayerChart(...)

## 3. Ageing over time: did faster-ageing regions see bigger falls in churn?

Sections 1–2 compare **levels** across regions. Here the question is about **change**: reallocation fell everywhere between the 2000s and 2020s, and the UK population aged (median age ≈ 38.7 in 2005 → 40.8 in 2022). If ageing drives the decline, regions that aged faster should see bigger falls. This uses the full **2003–2024** job-flows window (age data go back to 1991). **Northern Ireland is excluded** (no reliable job flows before 2011).

Two designs:
1. **Long differences:** change in reallocation between a 5-year early and late window, pooled from counts, against the rise in median age (or share 65+) between the same windows. Main: **2003–07 → 2020–24**; robustness: 2003–07 → 2015–19 (pre-COVID end) and 2004–08 → 2019–23.
2. **Panel with region and year fixed effects** (40 regions × 22 years): does a region's reallocation fall in years when it is older than usual, compared with other regions in the same year? Region effects absorb fixed differences (density, industry mix); year effects absorb the UK-wide decline, the cycle and the 2010/2023 spikes. Standard errors are clustered by region. Two stricter versions follow. The first excludes London. The second adds **density-specific year effects** (log density × year), so rural and urban regions can have their own trends: faster-ageing regions are mostly rural (r ≈ −0.8 with density), and rural churn could be falling for other reasons.

In [6]:
jf_all = reg[reg.analysis_window & (reg.itl2_code != "TLN0")].copy()          # 2003-24, NI excluded
age_all = pd.read_csv(DIR21 / "age_itl2.csv")
ma = age_all.pivot(index="itl2_code", columns="year", values="median_age")
logpwd = np.log(pwd)

def rate(yrs):
    gg = jf_all[jf_all.year.between(*yrs)].groupby("itl2_code")[["total_jc", "total_jd", "emp_avg"]].sum()
    return 100 * (gg.total_jc + gg.total_jd) / gg.emp_avg

def share65(yrs):
    s_ = age_all[age_all.year.between(*yrs)].groupby("itl2_code")[["population", "pop_65_plus"]].sum()
    return 100 * s_.pop_65_plus / s_.population

def resid(v, z):
    return v - np.polyval(np.polyfit(z, v, 1), z)

ld_rows, ld = [], None
for early, late in [((2003, 2007), (2020, 2024)), ((2003, 2007), (2015, 2019)), ((2004, 2008), (2019, 2023))]:
    d = pd.DataFrame({"d_realloc": rate(late) - rate(early),
                      "d_median": ma.loc[:, late[0]:late[1]].mean(axis=1) - ma.loc[:, early[0]:early[1]].mean(axis=1),
                      "d_65": share65(late) - share65(early)}).dropna().join(logpwd.rename("log_pwd")).join(names)
    o_ = d[~d.index.str.startswith("TLI")]
    ld_rows.append({"windows": f"{early[0]}–{str(early[1])[2:]} → {late[0]}–{str(late[1])[2:]}", "n": len(d),
                    "mean Δ realloc (pp)": d.d_realloc.mean(), "r Δmedian, all": np.corrcoef(d.d_realloc, d.d_median)[0, 1],
                    "r Δmedian, excl. London": np.corrcoef(o_.d_realloc, o_.d_median)[0, 1],
                    "partial | density, excl. London": np.corrcoef(resid(o_.d_realloc.values, o_.log_pwd.values), resid(o_.d_median.values, o_.log_pwd.values))[0, 1],
                    "r Δshare65, excl. London": np.corrcoef(o_.d_realloc, o_.d_65)[0, 1]})
    ld = d if ld is None else ld
print("(1) Long differences: change in reallocation vs rise in median age / share 65+")
print(pd.DataFrame(ld_rows).round(2).to_string(index=False))

# (2) panel: reallocation_it on age_it with region + year FE, clustered by region
pan = (jf_all[["itl2_code", "year", "reallocation_rate"]]
         .merge(age_all[["itl2_code", "year", "median_age", "share_65_plus"]], on=["itl2_code", "year"]))
pan["log_pwd"] = pan.itl2_code.map(logpwd)
YEARS = sorted(pan.year.unique())[1:]
for yy in YEARS:
    pan[f"pwd_x_{yy}"] = (pan.year == yy) * pan.log_pwd

def fe_coef(d, x, extra=()):
    X = np.column_stack([d[x].values] + [d[c].values for c in extra] +
                        [pd.get_dummies(d.itl2_code, drop_first=True, dtype=float), pd.get_dummies(d.year, drop_first=True, dtype=float), np.ones(len(d))])
    yv = d.reallocation_rate.values
    b = np.linalg.lstsq(X, yv, rcond=None)[0]; e = yv - X @ b; A = np.linalg.pinv(X.T @ X)
    G = pd.factorize(d.itl2_code)[0]; S = np.zeros((X.shape[1],) * 2)
    for gg in np.unique(G):
        sg = (X[G == gg] * e[G == gg][:, None]).sum(0); S += np.outer(sg, sg)
    n, k = X.shape; ng = G.max() + 1
    V = A @ S @ A * ng / (ng - 1) * (n - 1) / (n - k)
    return b[0], np.sqrt(V[0, 0]), n

fe_rows = []
DX = [f"pwd_x_{yy}" for yy in YEARS]
for x, lab in [("median_age", "Median age (per year of age)"), ("share_65_plus", "Share 65+ (per pp)")]:
    for spec, d, extra in [("Region + year FE", pan, ()), ("... excl. London", pan[~LONDON(pan)], ()),
                           ("... + density-specific year effects", pan, DX)]:
        b, se, n = fe_coef(d, x, extra)
        fe_rows.append({"measure": lab, "specification": spec, "coef (pp reallocation)": b, "se": se, "t": b / se, "N": n})
fe_tab = pd.DataFrame(fe_rows)
print("\n(2) Panel, region and year fixed effects, clustered by region")
print(fe_tab.round(3).to_string(index=False))

# how much of the UK decline could ageing account for?
uk_med = {yy: pooled_median(sy0[sy0.year == yy]) for yy, sy0 in [(2005, pd.read_csv(DIR21 / "age_single_year_itl2.csv").query("year == 2005")),
                                                                   (2022, pd.read_csv(DIR21 / "age_single_year_itl2.csv").query("year == 2022"))]}
d_uk_med = uk_med[2022] - uk_med[2005]
d_uk_realloc = rate((2020, 2024)).mean() - rate((2003, 2007)).mean()
b_hi = fe_tab.loc[(fe_tab.measure.str.startswith("Median")) & (fe_tab.specification == "Region + year FE"), "coef (pp reallocation)"].iloc[0]
b_lo = fe_tab.loc[(fe_tab.measure.str.startswith("Median")) & (fe_tab.specification.str.contains("density")), "coef (pp reallocation)"].iloc[0]
print(f"\nUK median age {uk_med[2005]:.1f} (2005) → {uk_med[2022]:.1f} (2022): +{d_uk_med:.1f} years. "
      f"Implied effect on reallocation: {b_lo * d_uk_med:+.1f} to {b_hi * d_uk_med:+.1f} pp, "
      f"vs an average regional fall of {d_uk_realloc:.1f} pp (2003–07 → 2020–24).")

(1) Long differences: change in reallocation vs rise in median age / share 65+
          windows  n  mean Δ realloc (pp)  r Δmedian, all  r Δmedian, excl. London  partial | density, excl. London  r Δshare65, excl. London
2003–07 → 2020–24 40                -6.46           -0.44                    -0.37                            -0.32                     -0.26
2003–07 → 2015–19 40                -6.04           -0.58                    -0.52                            -0.33                     -0.44
2004–08 → 2019–23 40                -5.21           -0.38                    -0.31                            -0.38                     -0.14

(2) Panel, region and year fixed effects, clustered by region
                     measure                       specification  coef (pp reallocation)    se      t   N
Median age (per year of age)                    Region + year FE                  -0.627 0.182 -3.438 880
Median age (per year of age)                    ... excl. London              

In [7]:
# chart: long differences, 2003-07 -> 2020-24
d = ld.reset_index().assign(group=lambda x: np.where(LONDON(x), "London", "Rest of UK"))
lr = ld_rows[0]
XS = alt.Scale(domain=[0, float(np.ceil(d.d_median.max() + 0.5))], zero=False, nice=False)
YS = alt.Scale(domain=[float(np.floor(d.d_realloc.min() - 0.5)), float(np.ceil(d.d_realloc.max() + 0.5))], zero=False, nice=False)
b = np.polyfit(d.d_median, d.d_realloc, 1); fx = np.array([d.d_median.min(), d.d_median.max()])
labs = place_labels(d, "d_median", "d_realloc", XS, YS, 520, 380, max_labels=6, max_len=22)
ldc = alt.layer(
    alt.Chart(pd.DataFrame({"v": [d.d_realloc.mean()]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS)),
    alt.Chart(pd.DataFrame({"x": [XS.domain[1]], "y": [d.d_realloc.mean()]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=10, color=TXT)
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS), text=alt.value(f"Average {d.d_realloc.mean():.1f} pp")),
    alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=ECO["turquoise"], strokeWidth=2, strokeDash=[6, 4])
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)),
    alt.Chart(d).mark_circle(size=70, opacity=0.85, stroke="white", strokeWidth=0.6).encode(
        x=alt.X("d_median:Q", scale=XS, title="Rise in median age, 2003–07 to 2020–24 (years)"),
        y=alt.Y("d_realloc:Q", scale=YS, title="Change in reallocation rate (percentage points)"),
        color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                        legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("d_median:Q", format=".1f", title="Rise in median age"),
                 alt.Tooltip("d_realloc:Q", format=".1f", title="Change in reallocation (pp)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
        .encode(x=alt.X("d_median:Q", scale=XS), y=alt.Y("d_realloc:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("Faster-ageing regions saw somewhat bigger falls in churn, partly because they are rural",
    subtitle=[f"ITL2 regions; change between 2003–07 and 2020–24 (5-year pooled rates). r = {lr['r Δmedian, all']:.2f} all regions; "
              f"{lr['r Δmedian, excl. London']:.2f} excl. London; {lr['partial | density, excl. London']:.2f} after controlling for density.",
              "Northern Ireland excluded (no reliable job flows before 2011). Dashed line = fit.", SOURCE]))
styles.save(ldc, str(FIG), "reallocation_ageing_change", width=520, height=380)
ldc.properties(width=520, height=380)

alt.LayerChart(...)

## 4. What this shows (and doesn't)

*(Numbers refer to the printed output above.)*

- **Older regions churn less**, by every measure (median age, share 65+, prime-age share). But age and density are tightly linked (r ≈ 0.8 excluding London), and **controlling for density removes about half of the correlation** (median age: −0.64 → −0.34). Age adds a little beyond rurality, not much.
- **Working-age population growth has the strongest link** (r ≈ 0.7 excluding London), and it largely **survives the density control** (≈ 0.6). It is linked most with **firm entry and exit**, as the labour-supply channel (Karahan, Pugsley & Şahin) predicts. But working-age growth and **net job growth** move together closely (r ≈ 0.8). Once net growth is also controlled for, nothing remains (≈ 0.05). The data can't tell whether more workers bring more firms and churn, or whether growing local economies attract workers. Both are plausible.
- **Ageing pace, 2011–24 only:** regions that aged faster have lower churn **levels** (partly surviving the density control, −0.73 → −0.54). But over this short window the **change** in churn is unrelated to ageing (r ≈ 0).
- **Ageing over the longer run (§3, 2003–24):** here the picture is clearer. Faster-ageing regions did see **bigger falls** in churn (long-difference r ≈ −0.3 to −0.5 excluding London, about −0.3 after controlling for density). In the fixed-effects panel, one extra year of median age goes with about **0.6 pp lower reallocation**. That falls to ~0.5 pp excluding London and to ~0.3 pp (not significant) once rural and urban regions are allowed their own trends. Faster-ageing regions are mostly rural, so part of the link is a rural–urban divergence rather than ageing as such.
- **How much of the decline could ageing explain?** The UK median age rose about 2 years (2005–2022). At 0.3–0.6 pp per year of age, that is **0.5–1.3 pp of a ~6.5 pp fall** in reallocation: roughly **a tenth to a fifth**. Ageing is a plausible contributor, not the main driver.
- **Caveats:**
  - **Measures are residence-based:** population is counted where people live, while job flows are counted where people work. Commuting into London and city centres blurs the link.
  - **Collinearity:** with 36–41 regions, density, age, migration and growth can't be separated cleanly.
  - **Next steps:** a within-region panel (reallocation vs lagged working-age growth, with region and year effects), and **internal migration** (`migration_itl2.csv`) as the channel that moves young workers into cities.

## 5. Slide charts

Three charts for a short slide sequence, kept light on method (details above):
1. **Younger regions, and those with growing working-age populations, churn more**: bars by quarter.
2. **...but age goes hand in hand with density: rural regions are older**: density vs median age, coloured by churn.
3. **Regions that aged faster tended to see bigger falls in churn**: region-level scatter, rise in median age vs % change in reallocation (2003–07 → 2020–24). Grouping into thirds hides this: the pattern is uneven, with the slow-ageing city regions falling least and the fast-ageing rural regions most. Checks (region level): slope −0.55 pp per year of age excl. London (p ≈ 0.03); stronger after controlling for starting churn; weaker without the three fastest-ageing regions.

**Combined alternative to slides 1–2:** one scatter, median age vs reallocation, coloured by density (`slide_1_2_age_density_churn`).

Groups are **quarters** (or thirds) of the non-London regions, with London shown separately; group rates come from **summed counts**.

In [8]:
# Slide 1: reallocation by median-age quarter and by working-age-growth quarter (2011-24 pooled; London separate)
SRC_S = "Sources: ONS Longitudinal Business Database (Secure Access, SN 9501); ONS, NRS and NISRA population estimates; authors' calculations."
nl = df[~LONDON(df)].copy()
def group_rate(codes):
    gg = jf[jf.year.between(*MAIN) & jf.itl2_code.isin(codes)][["total_jc", "total_jd", "emp_avg"]].sum()
    return 100 * (gg.total_jc + gg.total_jd) / gg.emp_avg

def quarter_bars(col, labels, title, xtitle):
    q = pd.qcut(nl[col], 4, labels=labels)
    rows = [{"group": "London", "rate": group_rate(df[LONDON(df)].itl2_code), "kind": "London"}]
    rows += [{"group": lab, "rate": group_rate(nl[q == lab].itl2_code), "kind": "Rest of UK"} for lab in labels]
    bd = pd.DataFrame(rows)
    order = labels + ["London"]
    base = alt.Chart(bd).encode(x=alt.X("group:N", sort=order, title=xtitle, axis=alt.Axis(labelAngle=0, labelLimit=120)),
                                y=alt.Y("rate:Q", scale=alt.Scale(domain=[0, 36]), title="Reallocation rate (%)" if col == "median_age" else None,
                                        axis=alt.Axis(labels=col == "median_age", ticks=col == "median_age")))
    return alt.layer(
        base.mark_bar(width=42).encode(color=alt.Color("kind:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-dark"]]), legend=None)),
        base.mark_text(dy=-8, fontSize=12, color=TXT).encode(text=alt.Text("rate:Q", format=".1f")),
    ).properties(width=300, height=260, title=alt.Title(title, fontSize=14, anchor="start"))

s1 = alt.hconcat(
    quarter_bars("median_age", ["Youngest", "2nd", "3rd", "Oldest"], "By median age", "Quarter of regions by median age"),
    quarter_bars("wa_growth", ["Slowest", "2nd", "3rd", "Fastest"], "By working-age population growth", "Quarter of regions by growth in 16–64 population"),
    spacing=30,
).properties(title=alt.Title("Younger regions, and those with growing working-age populations, have more job churn",
    subtitle=["Job reallocation rate (jobs created + destroyed, % of employment), pooled 2011–24. ITL2 regions outside London grouped into quarters (9 each);",
              "London's five regions shown separately.", SRC_S]))
styles.save(s1, str(FIG), "slide_1_age_churn_bars", width=300, height=260)
s1

alt.HConcatChart(...)

In [9]:
# Slide 1 trellis: reallocation by quarter of three age measures (36 regions outside London, 9 per quarter) + London; pooled 2011-24
TRELLIS = [("median_age", "Median age (years)", "", "{:.1f}"),
           ("share_65_plus", "Share of population aged 65+", "%", "{:.0f}"),
           ("wa_growth", "Working-age (16–64) population growth, a year", "%", "{:.1f}")]
lon_rate = group_rate(df[LONDON(df)].itl2_code)
avg_nl = group_rate(nl.itl2_code)
rows = []
for col, title, unit, fmt in TRELLIS:
    q, cuts = pd.qcut(nl[col], 4, labels=False, retbins=True)
    f = lambda v: fmt.format(v) + unit
    labs_q = [f"Under|{f(cuts[1])}", f"{f(cuts[1])}–|{f(cuts[2])}", f"{f(cuts[2])}–|{f(cuts[3])}", f"Over|{f(cuts[3])}"]
    for i in range(4):
        rows.append({"measure": title, "pos": i, "label": labs_q[i], "rate": group_rate(nl[q == i].itl2_code), "kind": "Rest of UK"})
    lon_val = df[LONDON(df)][col]
    rows.append({"measure": title, "pos": 5, "label": f"London|{f(lon_val.min())}–{f(lon_val.max())}", "rate": lon_rate, "kind": "London"})
    print(f"{title}: " + ", ".join(f"{l.replace('|', ' ')}: {r:.1f}" for l, r in zip(labs_q, [x['rate'] for x in rows[-5:-1]])))
tr = pd.DataFrame(rows)
tr["rate_lab"] = tr.rate.map(lambda v: f"{v:.1f}%")
print(f"London {lon_rate:.1f}; all 36 non-London regions {avg_nl:.1f}")

def panel_t(col, title, unit, first):
    d = tr[tr.measure == title]
    xq = alt.X("label:N", sort=list(d.sort_values("pos").label), title=None,
               axis=alt.Axis(labelAngle=0, labelFontSize=10.5, ticks=False,                 # middle groups unlabelled: only the end groups and London
                             labelExpr="indexof(datum.label, '–|') >= 0 ? '' : indexof(datum.label, 'London') == 0 ? 'London' : split(datum.label, '|')"))
    yq = alt.Y("rate:Q", scale=alt.Scale(domain=[0, 36], nice=False), title="Job reallocation rate (%)" if first else None,
               axis=alt.Axis(labels=first, ticks=first, values=[0, 10, 20, 30]))
    return alt.layer(
        alt.Chart(d).mark_bar(width=32).encode(x=xq, y=yq, color=alt.Color("kind:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-dark"]]), legend=None),
                                               tooltip=["measure:N", "label:N", alt.Tooltip("rate:Q", format=".1f")]),
        alt.Chart(d).mark_text(align="center", dx=0, dy=-8, fontSize=11, color=TXT).encode(x=xq, y=yq, text="rate_lab:N"),
    ).properties(width=245, height=240, title=alt.Title(title, fontSize=13, anchor="start", offset=8))

s1t = alt.hconcat(*[panel_t(col, t, u, i == 0) for i, (col, t, u, _) in enumerate(TRELLIS)], spacing=22).properties(title=alt.Title(
    "Younger regions, with growing working-age populations, have more job churn",
    subtitle=["Job reallocation rate (jobs created + destroyed, % of employment), pooled 2011–24. Blue bars: the 36 ITL2 regions outside London split into four",
              "equal groups (9 regions each) by each measure, from lowest to highest. Red bar: London's five regions, shown separately.", SRC_S]))
styles.save(s1t, str(FIG), "slide_1_age_trellis", width=245, height=240)
s1t

Median age (years): Under 40.6: 28.4, 40.6– 41.9: 26.9, 41.9– 43.8: 26.4, Over 43.8: 25.2
Share of population aged 65+: Under 18%: 28.4, 18%– 20%: 27.2, 20%– 22%: 26.3, Over 22%: 25.2
Working-age (16–64) population growth, a year: Under 0.1%: 25.2, 0.1%– 0.3%: 26.1, 0.3%– 0.5%: 26.9, Over 0.5%: 28.4
London 34.3; all 36 non-London regions 27.1


alt.HConcatChart(...)

In [10]:
# Slide 1, scatter version: one simple scatter per age measure against reallocation (pooled 2004-24, NI 2011-24)
SC = [("median_age", "Median age", "Median age (years)", "datum.label"),
      ("share_65_plus", "Share aged 65+", "Share of population aged 65+", "datum.label + '%'"),
      ("wa_growth", "Working-age population growth", "Growth in 16–64 population, % a year", "datum.label + '%'")]
# pooled 2004-24; Northern Ireland 2011-24 (job flows start in 2011), with its age measures over the same years
P0, P1 = 2004, 2024
start = lambda code: 2011 if code == "TLN0" else P0
ja = jf[jf.year.between(P0, P1)]                                   # jf already drops NI before 2011
gj = ja.groupby("itl2_code")[["total_jc", "total_jd", "emp_avg"]].sum()
aa = pd.read_csv(DIR21 / "age_itl2.csv")
aa = aa[aa.year.between(P0, P1) & (aa.year >= aa.itl2_code.map(start))]
sa = aa.groupby("itl2_code")[["population", "pop_65_plus"]].sum()
sya = pd.read_csv(DIR21 / "age_single_year_itl2.csv")
sya = sya[sya.year.between(P0, P1) & (sya.year >= sya.itl2_code.map(start))]
wa = aa.pivot(index="itl2_code", columns="year", values="pop_16_64")
dd = pd.DataFrame({"realloc": 100 * (gj.total_jc + gj.total_jd) / gj.emp_avg,
                   "share_65_plus": 100 * sa.pop_65_plus / sa.population,
                   "median_age": sya.groupby("itl2_code").apply(pooled_median, include_groups=False),
                   "wa_growth": pd.Series({c_: 100 * np.log(wa.loc[c_, P1] / wa.loc[c_, start(c_)]) / (P1 - start(c_)) for c_ in wa.index})})
dd = dd.join(names).reset_index(names="itl2_code")
dd["group"] = np.where(LONDON(dd), "London", dd.itl2_code.str[:3].map({"TLL": "Wales", "TLM": "Scotland", "TLN": "Northern Ireland"}).fillna("Rest of England"))
AREAS = ["London", "Rest of England", "Wales", "Scotland", "Northern Ireland"]
ACOLS = [ECO["red"], ECO["blue-light"], ECO["turquoise"], ECO["blue-dark"], ECO["yellow"]]
assert len(dd) == 41 and dd.notna().all().all()
print({k: round(np.corrcoef(dd[k], dd.realloc)[0, 1], 2) for k in ["median_age", "share_65_plus", "wa_growth"]})
YT = alt.Scale(domain=[20, 40], nice=False)
def age_scatter(col, title, xtitle, xexpr, first):
    lo, hi = dd[col].min(), dd[col].max(); pad = 0.06 * (hi - lo)
    XT = alt.Scale(domain=[float(lo - pad), float(hi + pad)], zero=False, nice=False)
    b = np.polyfit(dd[col], dd.realloc, 1); fx = np.array([lo, hi]); r = np.corrcoef(dd[col], dd.realloc)[0, 1]
    return alt.layer(
        alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=GREY, strokeWidth=1.8, opacity=0.7)
           .encode(x=alt.X("x:Q", scale=XT), y=alt.Y("y:Q", scale=YT)),
        alt.Chart(dd).mark_circle(size=60, opacity=0.9, stroke="white", strokeWidth=0.6).encode(
            x=alt.X(f"{col}:Q", scale=XT, title=xtitle, axis=alt.Axis(labelExpr=xexpr, tickCount=5)),
            y=alt.Y("realloc:Q", scale=YT, title="Job reallocation rate" if first else None,
                    axis=alt.Axis(labels=first, ticks=first, labelFlush=True, labelExpr="datum.label + '%'", values=[20, 25, 30, 35, 40])),
            color=alt.Color("group:N", scale=alt.Scale(domain=AREAS, range=ACOLS),
                            legend=alt.Legend(title=None, orient="bottom") if first else None),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip(f"{col}:Q", format=".1f", title=title),
                     alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    ).properties(width=250, height=250, title=alt.Title(title, subtitle=f"r = {r:.2f}", anchor="start", fontSize=13, subtitleFontSize=11))

s1s = alt.hconcat(*[age_scatter(c_, t, xt, ex, i == 0) for i, (c_, t, xt, ex) in enumerate(SC)], spacing=22).resolve_scale(color="shared").properties(
    title=alt.Title("Younger regions, with growing working-age populations, have more job churn",
    subtitle=["Each dot is an ITL2 region; job reallocation rate and age measures pooled over 2004–24 (Northern Ireland 2011–24). Grey line = fit across all regions.", SRC_S]))
styles.save(s1s, str(FIG), "slide_1_age_scatter_trellis", width=250, height=250)
s1s

{'median_age': np.float64(-0.77), 'share_65_plus': np.float64(-0.77), 'wa_growth': np.float64(0.67)}


alt.HConcatChart(...)

In [11]:
# Slide 2: density vs median age, coloured by reallocation
# all three measures pooled over 2018-24 (41 regions)
Y0, Y1 = 2018, 2024
j2 = jf[jf.year.between(Y0, Y1)].groupby("itl2_code")[["total_jc", "total_jd", "emp_avg"]].sum()
pw2 = pd.read_csv(DIR21 / "pop_weighted_density_itl2.csv").query("year >= @Y0 and year <= @Y1")
pw2 = pw2.assign(w=pw2.pop_weighted_density * pw2.population).groupby("itl2_code")[["w", "population"]].sum()
sy2 = pd.read_csv(DIR21 / "age_single_year_itl2.csv").query("year >= @Y0 and year <= @Y1")
d = pd.DataFrame({"realloc": 100 * (j2.total_jc + j2.total_jd) / j2.emp_avg, "pop_weighted_density": pw2.w / pw2.population,
                  "median_age": sy2.groupby("itl2_code").apply(pooled_median, include_groups=False)}).join(names).reset_index(names="itl2_code")
d["log_pwd"] = np.log(d.pop_weighted_density)
assert len(d) == 41 and d.notna().all().all()
print(f"2018–24: median age {d.median_age.min():.1f}–{d.median_age.max():.1f}; corr(density, age) {np.corrcoef(d.log_pwd, d.median_age)[0, 1]:.2f}; "
      f"corr(age, churn) {np.corrcoef(d.median_age, d.realloc)[0, 1]:.2f}; corr(density, churn) {np.corrcoef(d.log_pwd, d.realloc)[0, 1]:.2f}")
XS = alt.Scale(type="log", domain=[1000, 20000], nice=False)
YS = alt.Scale(domain=[float(np.floor(d.median_age.min() - 1)), float(np.ceil(d.median_age.max() + 1))], zero=False, nice=False)
W2, H2 = 560, 380
labs = place_labels(d.assign(group="x"), "pop_weighted_density", "median_age", XS, YS, W2, H2, xlog=True, max_labels=7, max_len=22)
r_dm = np.corrcoef(d.log_pwd, d.median_age)[0, 1]
bt = np.polyfit(d.log_pwd, d.median_age, 1); fxt = np.exp(np.linspace(d.log_pwd.min(), d.log_pwd.max(), 50))
s2 = alt.layer(
    alt.Chart(pd.DataFrame({"x": fxt, "y": np.polyval(bt, np.log(fxt))})).mark_line(color=GREY, strokeWidth=1.8, opacity=0.7)
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)),
    alt.Chart(d).mark_circle(size=130, stroke="white", strokeWidth=0.8, opacity=1).encode(
        x=alt.X("pop_weighted_density:Q", scale=XS, title="Population-weighted density (residents per km², log scale)",
                axis=alt.Axis(values=[1000, 2000, 5000, 10000, 20000], format=",")),
        y=alt.Y("median_age:Q", scale=YS, title="Median age (years)"),
        color=alt.Color("realloc:Q", scale=alt.Scale(range=["#e3f2fa", ECO["blue-light"], ECO["blue-dark"]]),
                        legend=alt.Legend(title="Job reallocation rate", orient="right", gradientLength=200, labelExpr="datum.label + '%'")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("pop_weighted_density:Q", format=",.0f", title="Density"),
                 alt.Tooltip("median_age:Q", format=".1f", title="Median age"), alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=10, color=TXT)
        .encode(x=alt.X("pop_weighted_density:Q", scale=XS), y=alt.Y("median_age:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("Rural regions are older, and older, rural regions churn less",
    subtitle=[f"ITL2 regions: population density against median age (correlation {r_dm:.2f}), coloured by job reallocation rate; all pooled over 2018–24.",
              "Darker = more churn; grey line = fit across all regions. The dense, young regions (London, big cities) churn most; sparse, older regions least.", SRC_S]))
styles.save(s2, str(FIG), "slide_2_density_age_churn", width=W2, height=H2)
s2.properties(width=W2, height=H2)

2018–24: median age 33.1–48.0; corr(density, age) -0.88; corr(age, churn) -0.75; corr(density, churn) 0.78


alt.LayerChart(...)

In [12]:
# Slide 1+2 combined: median age vs reallocation, coloured by population density (2011-24)
d = df.assign(pop_weighted_density=np.exp(df.log_pwd))
W12, H12 = 560, 380
XS = alt.Scale(domain=[32, 48], zero=False, nice=False)
YS = alt.Scale(domain=[21, 39], zero=False, nice=False)
b = np.polyfit(d.median_age, d.realloc, 1); fx = np.array([d.median_age.min(), d.median_age.max()])
r_ar = np.corrcoef(d.median_age, d.realloc)[0, 1]; r_ad = np.corrcoef(d.median_age, d.log_pwd)[0, 1]
labs = place_labels(d.assign(group="x"), "median_age", "realloc", XS, YS, W12, H12, max_labels=8, max_len=22, pad=2)
s12 = alt.layer(
    alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=GREY, strokeWidth=2, strokeDash=[6, 4])
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)),
    alt.Chart(d).mark_circle(size=130, stroke="#7a8a94", strokeWidth=0.5, opacity=1).encode(
        x=alt.X("median_age:Q", scale=XS, title="Median age, 2011–24 (years)"),
        y=alt.Y("realloc:Q", scale=YS, title="Job reallocation rate, 2011–24 (% of employment)"),
        color=alt.Color("pop_weighted_density:Q", scale=alt.Scale(type="log", domain=[1500, 15000], range=["#cde8f6", ECO["blue-light"], ECO["blue-dark"]]),
                        legend=alt.Legend(title=["Population density", "(residents per km²)"], orient="right", gradientLength=200,
                                          values=[2000, 5000, 10000], format=",")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("median_age:Q", format=".1f", title="Median age"),
                 alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)"), alt.Tooltip("pop_weighted_density:Q", format=",.0f", title="Density")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=10, color=TXT)
        .encode(x=alt.X("median_age:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("Younger regions have more job churn, and they are also the denser ones",
    subtitle=[f"ITL2 regions: median age against job reallocation rate (jobs created + destroyed, % of employment), pooled 2011–24 (correlation {r_ar:.2f}).",
              f"Colour = population density (population-weighted, log scale). Younger regions are much denser (correlation {r_ad:.2f}): dense, young cities churn most.", SRC_S]))
styles.save(s12, str(FIG), "slide_1_2_age_density_churn", width=W12, height=H12)
s12.properties(width=W12, height=H12)

alt.LayerChart(...)

In [13]:
# Slide 3: region-level scatter - rise in median age vs % fall in reallocation, 2003-07 -> 2020-24 (NI excluded)
s3d = pd.DataFrame({"r0": rate((2003, 2007)), "r1": rate((2020, 2024))}).join(ld[["d_median", "itl2_name"]], how="inner").reset_index(names="itl2_code")
s3d["pct"] = 100 * (s3d.r1 / s3d.r0 - 1)
s3d["group"] = np.where(LONDON(s3d), "London", "Rest of UK")
nl3 = s3d[~LONDON(s3d)]
r3_all, r3_ex = np.corrcoef(s3d.d_median, s3d.pct)[0, 1], np.corrcoef(nl3.d_median, nl3.pct)[0, 1]
print(f"% change in reallocation vs rise in median age: r = {r3_all:.2f} all, {r3_ex:.2f} excl. London")

HIGHLIGHT = {"West Midlands": "Slow-ageing city regions", "Greater Manchester": "Slow-ageing city regions",
             "South Yorkshire": "Slow-ageing city regions", "Merseyside": "Slow-ageing city regions",
             "Highlands and Islands": "Fast-ageing rural regions", "Southern Scotland": "Fast-ageing rural regions",
             "Cumbria": "Fast-ageing rural regions", "North Yorkshire": "Fast-ageing rural regions"}
s3d["hl"] = s3d.itl2_name.map(HIGHLIGHT).fillna(s3d.group)
HORDER = ["Slow-ageing city regions", "Fast-ageing rural regions", "London", "Rest of UK"]
HCOLS = [ECO["blue-dark"], ECO["orange"], ECO["red"], "#9fd3ee"]
W3, H3 = 540, 380
XS = alt.Scale(domain=[0, 6.5], zero=False, nice=False)
YS = alt.Scale(domain=[-35, -5], zero=False, nice=False)
b = np.polyfit(s3d.d_median, s3d.pct, 1); fx = np.array([s3d.d_median.min(), s3d.d_median.max()])
labs = place_labels(s3d, "d_median", "pct", XS, YS, W3, H3, max_labels=None, max_len=None, pad=2, candidates=list(HIGHLIGHT))
missing = set(HIGHLIGHT) - set(labs.itl2_name) if len(labs) else set(HIGHLIGHT)
print("highlighted but unlabelled (no clear space):", missing or "none")
# hand-placed labels for the two crowded city regions (above / below their dots)
MANUAL = {"Greater Manchester": dict(align="left", dx=-6, dy=-12), "South Yorkshire": dict(align="center", dx=0, dy=14)}
man = s3d[s3d.itl2_name.isin(MANUAL)].assign(short=lambda x: x.itl2_name).reset_index(drop=True)
s3 = alt.layer(
    alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=GREY, strokeWidth=2, strokeDash=[6, 4])
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)),
    alt.Chart(s3d).mark_circle(size=90, opacity=1, stroke="white", strokeWidth=0.8).encode(
        x=alt.X("d_median:Q", scale=XS, title="Rise in median age, 2003–07 to 2020–24 (years)"),
        y=alt.Y("pct:Q", scale=YS, title="Change in job reallocation rate", axis=alt.Axis(labelExpr="datum.label + '%'")),
        color=alt.Color("hl:N", scale=alt.Scale(domain=HORDER, range=HCOLS), legend=alt.Legend(title=None, orient="right", direction="vertical")),
        order=alt.Order("hl:N", sort="descending"),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("d_median:Q", format=".1f", title="Rise in median age (yrs)"),
                 alt.Tooltip("r0:Q", format=".1f", title="Reallocation 2003–07 (%)"), alt.Tooltip("r1:Q", format=".1f", title="Reallocation 2020–24 (%)"),
                 alt.Tooltip("pct:Q", format=".0f", title="Change (%)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=10, color=TXT)
        .encode(x=alt.X("d_median:Q", scale=XS), y=alt.Y("pct:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
    *[alt.Chart(man.iloc[[i]]).mark_text(fontSize=10, color=TXT, **MANUAL[r["itl2_name"]])
        .encode(x=alt.X("d_median:Q", scale=XS), y=alt.Y("pct:Q", scale=YS), text="short:N") for i, r in man.reset_index(drop=True).iterrows()],
).properties(title=alt.Title("Regions that aged faster tended to see bigger falls in job churn",
    subtitle=[f"ITL2 regions: rise in median age against the % change in the job reallocation rate, 2003–07 to 2020–24 (correlation {r3_all:.2f}; {r3_ex:.2f} excl. London).",
              "Slow-ageing city regions saw the smallest falls; fast-ageing rural regions some of the largest. Northern Ireland excluded (no reliable data before 2011).", SRC_S]))
styles.save(s3, str(FIG), "slide_3_ageing_churn_fall", width=W3, height=H3)
s3.properties(width=W3, height=H3)

% change in reallocation vs rise in median age: r = -0.52 all, -0.45 excl. London
highlighted but unlabelled (no clear space): {'South Yorkshire', 'Greater Manchester'}


alt.LayerChart(...)